# Weather Exposure Builder (Legacy)

**Source script:** `build_feline_weather_exposures.py`

Notebook mirror for submission traceability.

In [ ]:
from __future__ import annotations

from pathlib import Path
from datetime import timedelta, date
import time
import math
import requests
import pandas as pd




MAX_RETRIES = 5

BASE_DIR = Path(".")

CLINICAL_FILE = BASE_DIR / "Data_set_cats for analysis.xlsx"
CLINICAL_SHEET = "Weather data"

EXPOSURE_OUTPUT_FILE = BASE_DIR / "Weather_Exposure_Data.csv"


COL_CASE_ID = "ID"
COL_ZIP = "zip code"
COL_TOWN = "town/village"
COL_STATE = "state"
COL_DATE = "date of presentation"




GEOCODE_CACHE_FILE = BASE_DIR / "geocoded_locations.csv"
WEATHER_CACHE_FILE = BASE_DIR / "weather_daily_cache.csv"





NOMINATIM_URL = "https://nominatim.openstreetmap.org/search"
USER_AGENT = "feline-weather-thesis/1.0 (omar@alneser.de)"

OPEN_METEO_URL = "https://archive-api.open-meteo.com/v1/archive"

DAILY_VARS = [
    "temperature_2m_max",
    "temperature_2m_min",
    "temperature_2m_mean",
    "wind_speed_10m_max",
    "shortwave_radiation_sum",
    "precipitation_sum",
    "rain_sum",
]

TIMEZONE = "auto"






def load_weather_sheet(path: Path, sheet_name: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Clinical file not found: {path}")

    print(f"Loading clinical 'Weather data' from {path} (sheet={sheet_name})")
    df = pd.read_excel(path, sheet_name=sheet_name)

    required_cols = [COL_CASE_ID, COL_ZIP, COL_TOWN, COL_STATE, COL_DATE]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"Required column '{col}' not found in sheet '{sheet_name}'.")

    df = df.copy()
    df[COL_DATE] = pd.to_datetime(df[COL_DATE])


    def clean_zip(x):
        if pd.isna(x):
            return ""
        if isinstance(x, float) or isinstance(x, int):
            return str(int(x))
        return str(x).strip()

    df[COL_ZIP] = df[COL_ZIP].apply(clean_zip)

    print("Weather data shape:", df.shape)
    return df


def safe_str(x) -> str:
    """Convert any value (float, NaN, None, int, str) into a clean string."""
    if pd.isna(x):
        return ""
    if isinstance(x, float) or isinstance(x, int):
        return str(int(x))
    return str(x).strip()


def build_location_key(zip_code, town, state) -> str:
    zip_str = safe_str(zip_code)
    town_str = safe_str(town)
    state_str = safe_str(state)
    return f"{zip_str}|{town_str}|{state_str}"


def geocode_location(zip_code, town, state):
    """
    Use Nominatim to geocode (zip, town, state) → (lat, lon).
    Robust to floats, NaNs, etc.
    """
    zip_code = safe_str(zip_code)
    town = safe_str(town)
    state = safe_str(state)

    query_parts = []
    if zip_code:
        query_parts.append(zip_code)
    if town:
        query_parts.append(town)
    if state:
        query_parts.append(state)

    query_parts.append("Germany")

    query = ", ".join(query_parts)

    params = {
        "q": query,
        "format": "json",
        "limit": 1,
    }
    headers = {
        "User-Agent": USER_AGENT,
    }

    print(f"  Geocoding '{query}' ...")
    try:
        resp = requests.get(NOMINATIM_URL, params=params, headers=headers, timeout=30)
    except requests.exceptions.RequestException as e:
        print(f"    Geocoding request failed for '{query}': {e}")
        return None, None

    if resp.status_code != 200:
        print(f"    Geocoding error: HTTP {resp.status_code}")
        return None, None

    try:
        data = resp.json()
    except ValueError as e:
        print(f"    Failed to parse geocoding JSON for '{query}': {e}")
        return None, None

    if not data:
        print("    No geocoding result.")
        return None, None

    lat = float(data[0]["lat"])
    lon = float(data[0]["lon"])
    print(f"    → lat={lat:.5f}, lon={lon:.5f}")
    return lat, lon


def get_date_range(weather_df: pd.DataFrame, lag_buffer_days: int = 14) -> tuple[date, date]:
    min_date = weather_df[COL_DATE].min()
    max_date = weather_df[COL_DATE].max()
    start_date = (min_date - timedelta(days=lag_buffer_days)).date()
    end_date = max_date.date()
    print(f"Study presentation dates: {min_date.date()} to {max_date.date()}")
    print(f"Weather query range with buffer: {start_date} to {end_date}")
    return start_date, end_date


def fetch_openmeteo_daily(lat: float, lon: float, start_date: date, end_date: date) -> pd.DataFrame:
    """
    Call Open-Meteo Historical Weather API for a single location.
    Includes retry logic with backoff to handle 429 (rate limit) and 5xx errors.
    """
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date.isoformat(),
        "end_date": end_date.isoformat(),
        "daily": ",".join(DAILY_VARS),
        "timezone": TIMEZONE,
    }

    for attempt in range(1, MAX_RETRIES + 1):
        resp = requests.get(OPEN_METEO_URL, params=params, timeout=60)


        if resp.status_code == 200:
            data = resp.json()
            if "daily" not in data or "time" not in data["daily"]:
                raise ValueError(f"No 'daily' data returned for lat={lat}, lon={lon}: {data}")
            daily = data["daily"]
            dates = daily["time"]

            records = []
            for i, dstr in enumerate(dates):
                rec = {"date": pd.to_datetime(dstr).date()}
                for var in DAILY_VARS:
                    if var in daily:
                        rec[var] = daily[var][i]
                records.append(rec)

            return pd.DataFrame.from_records(records)


        if resp.status_code == 429:
            wait = 60 * attempt
            print(
                f"Open-Meteo 429 (rate limit) for lat={lat}, lon={lon}, "
                f"attempt {attempt}/{MAX_RETRIES}. Waiting {wait} seconds..."
            )
            time.sleep(wait)
            continue


        if 500 <= resp.status_code < 600:
            wait = 5 * attempt
            print(
                f"Open-Meteo server error {resp.status_code} for lat={lat}, lon={lon}, "
                f"attempt {attempt}/{MAX_RETRIES}. Waiting {wait} seconds..."
            )
            time.sleep(wait)
            continue


        raise RuntimeError(
            f"Open-Meteo error for lat={lat}, lon={lon}: "
            f"HTTP {resp.status_code}, body={resp.text[:300]}"
        )


    raise RuntimeError(
        f"Open-Meteo failed for lat={lat}, lon={lon} after {MAX_RETRIES} attempts."
    )







def load_geocode_cache(path: Path) -> pd.DataFrame:
    if path.exists():
        cache = pd.read_csv(path)
        expected_cols = ["location_key", "zip_code", "town", "state", "latitude", "longitude"]
        for col in expected_cols:
            if col not in cache.columns:
                raise ValueError(f"Geocode cache at {path} is missing column: {col}")
        return cache
    else:
        return pd.DataFrame(columns=["location_key", "zip_code", "town", "state", "latitude", "longitude"])


def save_geocode_cache(cache: pd.DataFrame, path: Path) -> None:
    cache.to_csv(path, index=False)


def build_locations_table(weather_df: pd.DataFrame) -> pd.DataFrame:
    """
    Build a table of unique (zip, town, state) locations with lat/lon,
    using a persistent CSV cache (GEOCODE_CACHE_FILE) so that already
    geocoded locations are not queried again.
    """
    tmp = weather_df[[COL_ZIP, COL_TOWN, COL_STATE]].copy()
    tmp["location_key"] = tmp.apply(
        lambda r: build_location_key(r[COL_ZIP], r[COL_TOWN], r[COL_STATE]), axis=1
    )
    unique_locs = tmp.drop_duplicates("location_key")
    print(f"Unique (zip, town, state) combinations in data: {unique_locs.shape[0]}")

    cache = load_geocode_cache(GEOCODE_CACHE_FILE)
    existing_keys = set(cache["location_key"])
    todo = unique_locs[~unique_locs["location_key"].isin(existing_keys)]

    print(f"Locations already in geocode cache: {len(existing_keys)}")
    print(f"Locations to geocode now: {todo.shape[0]}")

    new_rows = []
    for _, row in todo.iterrows():
        zip_code = row[COL_ZIP]
        town = row[COL_TOWN]
        state = row[COL_STATE]
        location_key = row["location_key"]

        lat, lon = geocode_location(zip_code, town, state)

        time.sleep(1.0)

        new_row = {
            "location_key": location_key,
            "zip_code": safe_str(zip_code),
            "town": safe_str(town),
            "state": safe_str(state),
            "latitude": lat,
            "longitude": lon,
        }


        cache = pd.concat([cache, pd.DataFrame([new_row])], ignore_index=True)
        save_geocode_cache(cache, GEOCODE_CACHE_FILE)

    print("Geocoded locations (from cache + new):")
    print(cache.head())
    return cache







def load_weather_cache(path: Path) -> pd.DataFrame:
    """
    Load the persistent weather cache. If the file does not exist or is empty,
    return an empty dataframe with the expected columns.
    """
    cols = ["location_key", "date"] + DAILY_VARS
    if path.exists():
        try:
            df = pd.read_csv(path)
        except pd.errors.EmptyDataError:

            print(f"Weather cache file {path} is empty. Initializing empty cache.")
            return pd.DataFrame(columns=cols)
        if df.empty and len(df.columns) == 0:

            print(f"Weather cache file {path} has no usable columns. Initializing empty cache.")
            return pd.DataFrame(columns=cols)
        if "date" in df.columns:
            df["date"] = pd.to_datetime(df["date"])
        return df
    else:
        return pd.DataFrame(columns=cols)


def save_weather_cache(cache: pd.DataFrame, path: Path) -> None:
    cache.to_csv(path, index=False)


def build_daily_weather(weather_df: pd.DataFrame, loc_df: pd.DataFrame) -> pd.DataFrame:
    start_date, end_date = get_date_range(weather_df, lag_buffer_days=14)


    weather_cache = load_weather_cache(WEATHER_CACHE_FILE)

    all_daily = []


    loc_df = loc_df.copy()
    num_locs = loc_df.shape[0]

    for idx, row in enumerate(loc_df.itertuples(index=False), start=1):
        location_key = row.location_key
        lat = row.latitude
        lon = row.longitude

        if pd.isna(lat) or pd.isna(lon):
            print(f"Skipping location_key={location_key} (no coordinates).")
            continue


        loc_cache = weather_cache[weather_cache["location_key"] == location_key].copy()
        loc_cache_dates = set(loc_cache["date"].dt.date) if not loc_cache.empty else set()

        needed_dates = pd.date_range(start=start_date, end=end_date, freq="D").date
        missing_dates = [d for d in needed_dates if d not in loc_cache_dates]

        if missing_dates:
            print(f"[{idx}/{num_locs}] Fetching weather for {location_key}: lat={lat}, lon={lon}")
            daily_df = fetch_openmeteo_daily(lat, lon, start_date, end_date)
            daily_df["location_key"] = location_key


            daily_df["date"] = pd.to_datetime(daily_df["date"])
            weather_cache = pd.concat([weather_cache, daily_df], ignore_index=True)
            save_weather_cache(weather_cache, WEATHER_CACHE_FILE)

            all_daily.append(daily_df)

            time.sleep(1.0)
        else:
            print(f"[{idx}/{num_locs}] Using cached weather for {location_key}")
            all_daily.append(loc_cache)

    if not all_daily:
        raise RuntimeError("No daily weather data fetched or found in cache. Check geocoding / API access.")


    weather_daily = pd.concat(all_daily, ignore_index=True)
    weather_daily["date"] = pd.to_datetime(weather_daily["date"])
    mask = (weather_daily["date"].dt.date >= start_date) & (weather_daily["date"].dt.date <= end_date)
    weather_daily = weather_daily[mask].copy()


    weather_daily = weather_daily.rename(
        columns={
            "temperature_2m_mean": "temp_mean",
            "wind_speed_10m_max": "wind_max",
            "shortwave_radiation_sum": "radiation_sum",
            "rain_sum": "rain_sum",
        }
    )

    print("Combined daily weather shape:", weather_daily.shape)
    return weather_daily






def add_temp_anomaly(weather_daily: pd.DataFrame) -> pd.DataFrame:
    df = weather_daily.copy()
    df["month"] = df["date"].dt.month

    clim = (
        df.groupby(["location_key", "month"])["temp_mean"]
        .agg(["mean", "std"])
        .reset_index()
        .rename(columns={"mean": "clim_mean", "std": "clim_std"})
    )
    df = df.merge(clim, on=["location_key", "month"], how="left")
    df["Temp_Anomaly"] = (df["temp_mean"] - df["clim_mean"]) / df["clim_std"]
    return df


def build_case_exposures(weather_df: pd.DataFrame, loc_df: pd.DataFrame, weather_daily: pd.DataFrame) -> pd.DataFrame:
    cases = weather_df.copy()
    cases["location_key"] = cases.apply(
        lambda r: build_location_key(r[COL_ZIP], r[COL_TOWN], r[COL_STATE]), axis=1
    )
    cases["Date_only"] = cases[COL_DATE].dt.date

    daily = add_temp_anomaly(weather_daily)
    daily["date_only"] = daily["date"].dt.date


    daily_grouped = {loc: g.sort_values("date_only") for loc, g in daily.groupby("location_key")}

    exposure_rows = []

    for _, row in cases.iterrows():
        case_id = row[COL_CASE_ID]
        loc_key = row["location_key"]
        pres_date = row["Date_only"]

        loc_weather = daily_grouped.get(loc_key)
        if loc_weather is None or loc_weather.empty:
            exposure_rows.append({
                "Case_ID": case_id,
                "Date": row[COL_DATE],
                "Temp_Lag0": pd.NA,
                "Temp_Lag1": pd.NA,
                "Temp_Lag2": pd.NA,
                "Wind_Max_3d": pd.NA,
                "Radiation_7d_Peak": pd.NA,
                "Rain_14d_Sum": pd.NA,
                "Temp_Delta_24h": pd.NA,
                "Temp_Anomaly": pd.NA,
            })
            continue

        def select_range(days_back_start: int, days_back_end: int) -> pd.DataFrame:
            start = pres_date - timedelta(days=days_back_end)
            end = pres_date - timedelta(days=days_back_start)
            mask = (loc_weather["date_only"] >= start) & (loc_weather["date_only"] <= end)
            return loc_weather[mask]


        lag0 = loc_weather[loc_weather["date_only"] == pres_date]
        lag1 = loc_weather[loc_weather["date_only"] == pres_date - timedelta(days=1)]
        lag2 = loc_weather[loc_weather["date_only"] == pres_date - timedelta(days=2)]

        def val_or_na(df_lag, col):
            return df_lag.iloc[0][col] if not df_lag.empty else pd.NA

        temp_lag0 = val_or_na(lag0, "temp_mean")
        temp_lag1 = val_or_na(lag1, "temp_mean")
        temp_lag2 = val_or_na(lag2, "temp_mean")

        last3 = select_range(0, 2)
        wind_3d_max = last3["wind_max"].max() if not last3.empty else pd.NA

        last7 = select_range(0, 6)
        rad_7d_peak = last7["radiation_sum"].max() if not last7.empty else pd.NA

        last14 = select_range(0, 13)
        rain_14d_sum = last14["rain_sum"].sum() if not last14.empty else pd.NA

        if not lag0.empty and not lag1.empty:
            temp_delta_24h = lag0.iloc[0]["temp_mean"] - lag1.iloc[0]["temp_mean"]
        else:
            temp_delta_24h = pd.NA

        temp_anomaly = val_or_na(lag0, "Temp_Anomaly")

        exposure_rows.append({
            "Case_ID": case_id,
            "Date": row[COL_DATE],
            "Temp_Lag0": temp_lag0,
            "Temp_Lag1": temp_lag1,
            "Temp_Lag2": temp_lag2,
            "Wind_Max_3d": wind_3d_max,
            "Radiation_7d_Peak": rad_7d_peak,
            "Rain_14d_Sum": rain_14d_sum,
            "Temp_Delta_24h": temp_delta_24h,
            "Temp_Anomaly": temp_anomaly,
        })

    exposure_df = pd.DataFrame(exposure_rows)
    print("Case-level exposure table shape:", exposure_df.shape)
    return exposure_df






def main():
    print("=== STEP 1: Load feline 'Weather data' sheet ===")
    weather_df = load_weather_sheet(CLINICAL_FILE, CLINICAL_SHEET)

    print("\n=== STEP 2: Geocode all unique (zip, town, state) ===")
    loc_df = build_locations_table(weather_df)

    print("\n=== STEP 3: Fetch daily weather for each location ===")
    weather_daily = build_daily_weather(weather_df, loc_df)

    print("\n=== STEP 4: Compute per-case exposure features ===")
    exposure_df = build_case_exposures(weather_df, loc_df, weather_daily)

    print(f"\n=== STEP 5: Save Weather_Exposure_Data to {EXPOSURE_OUTPUT_FILE} ===")
    exposure_df.to_csv(EXPOSURE_OUTPUT_FILE, index=False)

    print("Done. Preview:")
    print(exposure_df.head())


if __name__ == "__main__":
    main()
